# Workflow 3 — Bootstrap sensitivity tables & paper figures

Produces:
1. Table of R(p) ± CI with p-values, for each selection
2. Figure: single-selection significance plot (per selection)
3. Figure: multi-selection grouped comparison
4. LaTeX-ready table for the paper
5. Redshift-evolution figure across snapshots

Uses `load_and_analyze` from `src/analysis_pipeline.py` instead of a
local re-implementation, and the bootstrap/null machinery from
`src/sensitivity_bootstrap.py`.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis_pipeline import load_and_analyze
from src.sensitivity_bootstrap import (
    sensitivity_table,
    plot_sensitivity,
    plot_multi_selection,
)

## Config

In [2]:
PARAM_FILE = (
    "../CAMELS-master/docs/params/IllustrisTNG/"
    "CosmoAstroSeed_IllustrisTNG_L25n256_LH.txt"
)
OUTPUT_DIR = "../outputs"
FIG_DIR    = "../plots"

PARAMS = [
    "Omega_m", "sigma_8",
    "A_SN1",   "A_AGN1",
    "A_SN2",   "A_AGN2",
]

# Quick mode for iterating on the notebook; switch to False for the
# final paper run (this can take a while: len(SELECTIONS) * len(PARAMS)
# * (N_BOOT + N_NULL) resamples).
QUICK = False
N_BOOT, N_NULL = (200, 200) if QUICK else (5000, 5000)

print(f"QUICK={QUICK}  N_BOOT={N_BOOT}  N_NULL={N_NULL}")

# Selections to compare.
# NOTE: generate_knn_active.py's output filename only encodes
# (selection, snap, mass_cut) -- NOT top_fraction. If you've generated
# more than one top_fraction for the same selection/snap/mass_cut
# (e.g. luminosity top-10% AND top-2% at snap50/M1e6), they will have
# overwritten each other on disk and "L2" below may not actually be
# the top-2% run. Re-run workflow 2 with a fixed output filename
# (including top_fraction) before trusting this for the paper table.
SELECTIONS = {
    "M1e6":   f"{OUTPUT_DIR}/knn_snap50_M1e+06.npz",
    "M1e7":   f"{OUTPUT_DIR}/knn_snap50_M1e+07.npz",
    "M1e8":   f"{OUTPUT_DIR}/knn_snap50_M1e+08.npz",
    "fEdd10": f"{OUTPUT_DIR}/knn_fedd_snap50_M1e+06.npz",
    "L10":     f"{OUTPUT_DIR}/knn_luminosity_snap50_M1e+06.npz",
}

# Snapshot -> redshift mapping, used for the evolution figure in
# section 4. Keyed by snap number so labels stay correct even if a
# snapshot is missing or the dict is reordered (unlike the old
# positional [3,1,0][:len(...)] slicing).
SNAP_TO_Z = {
    32: 3,
    50: 1,
    90: 0,
}

QUICK=False  N_BOOT=5000  N_NULL=5000


## 1. Per-selection tables

In [3]:
all_tables = {}

for label, path in SELECTIONS.items():

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")

    try:
        result = load_and_analyze(path, params=PARAMS, params_file=PARAM_FILE)
    except FileNotFoundError as e:
        print(f"  [skip] file not found: {e}")
        continue

    residuals, theta = result["residuals"], result["theta"]

    df = sensitivity_table(
        residuals, theta,
        params=PARAMS,
        n_boot=N_BOOT,
        n_null=N_NULL,
    )

    all_tables[label] = df

    print(df.to_string(index=False, float_format="%.5f"))
    print()

    # Individual significance plot
    fig, ax = plot_sensitivity(df, title=label)
    fig.savefig(f"{FIG_DIR}/sensitivity_{label}.pdf", dpi=300)
    fig.savefig(f"{FIG_DIR}/sensitivity_{label}.png", dpi=150)
    plt.close(fig)
    print(f"  -> {FIG_DIR}/sensitivity_{label}.pdf")


  M1e6


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00725 0.00420 0.01008     0.00261  0.00000         True
  sigma_8 0.00492 0.00321 0.00686     0.00262  0.00000         True
    A_SN1 0.00435 0.00267 0.00659     0.00258  0.00020         True
   A_AGN1 0.00221 0.00089 0.00439     0.00263  0.11400        False
    A_SN2 0.00191 0.00087 0.00414     0.00263  0.21680        False
   A_AGN2 0.00057 0.00049 0.00297     0.00257  0.91780        False

  -> ../plots/sensitivity_M1e6.pdf

  M1e7


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.01112 0.00876 0.01400     0.00404  0.00000         True
  sigma_8 0.01104 0.00783 0.01457     0.00409  0.00000         True
    A_SN1 0.00718 0.00435 0.01108     0.00409  0.00020         True
   A_AGN1 0.00190 0.00123 0.00540     0.00413  0.58140        False
    A_SN2 0.00404 0.00171 0.00777     0.00404  0.05060        False
   A_AGN2 0.00490 0.00211 0.00866     0.00408  0.01520         True

  -> ../plots/sensitivity_M1e7.pdf

  M1e8


...

parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00980 0.00722 0.01293     0.00446  0.00000         True
  sigma_8 0.00847 0.00475 0.01242     0.00449  0.00020         True
    A_SN1 0.00315 0.00168 0.00747     0.00452  0.23740        False
   A_AGN1 0.00264 0.00159 0.00660     0.00454  0.39800        False
    A_SN2 0.00243 0.00163 0.00614     0.00450  0.46060        False
   A_AGN2 0.00179 0.00128 0.00568     0.00455  0.72480        False

  -> ../plots/sensitivity_M1e8.pdf

  fEdd10


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00899 0.00539 0.01242     0.00318  0.00000         True
  sigma_8 0.00191 0.00131 0.00489     0.00323  0.36720        False
    A_SN1 0.00493 0.00334 0.00750     0.00323  0.00060         True
   A_AGN1 0.00210 0.00095 0.00501     0.00321  0.28620        False
    A_SN2 0.00284 0.00151 0.00556     0.00322  0.09840        False
   A_AGN2 0.00132 0.00095 0.00436     0.00316  0.67500        False

  -> ../plots/sensitivity_fEdd10.pdf

  L10


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00839 0.00498 0.01170     0.00318  0.00000         True
  sigma_8 0.00265 0.00140 0.00531     0.00325  0.14540        False
    A_SN1 0.00337 0.00184 0.00616     0.00320  0.03380         True
   A_AGN1 0.00181 0.00094 0.00477     0.00325  0.41540        False
    A_SN2 0.00235 0.00129 0.00504     0.00321  0.21540        False
   A_AGN2 0.00122 0.00078 0.00427     0.00318  0.72520        False

  -> ../plots/sensitivity_L10.pdf


In [4]:
from src.selection_bias import diagnose_activity_cut_bias, nbh_corrected_sensitivity
from src.parameter_sensitivity import load_params

theta_all = load_params(np.arange(1000), PARAM_FILE)

# fEdd10 was never checked for the nbh-size bias in Workflow 2 — only
# luminosity was. Confirm before deciding whether it needs correcting too.
result_fedd = load_and_analyze(
    SELECTIONS["fEdd10"], params=PARAMS, params_file=PARAM_FILE,
)
fedd_diag = diagnose_activity_cut_bias(
    sim_ids=result_fedd["sim_ids"], nbh=result_fedd["nbh"],
    theta_all=theta_all, params=PARAMS,
)
print("fEdd10 activity-cut size bias:")
print(fedd_diag.to_string(index=False))

fEdd10 activity-cut size bias:
parameter  spearman_rho  spearman_p  bias_flag
  Omega_m      0.980311    0.000000       True
  sigma_8      0.143655    0.000005       True
    A_SN1     -0.078378    0.013166      False
   A_AGN1      0.003234    0.918642      False
    A_SN2      0.018618    0.556496      False
   A_AGN2     -0.007503    0.812674      False


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


In [5]:
raw_df, corrected_df, theta_resid = nbh_corrected_sensitivity(
    residuals=result["residuals"],
    theta=result["theta"],
    nbh=result["nbh"],
    params=PARAMS,
    n_boot=N_BOOT, n_null=N_NULL,
)

print("L10 raw:")
print(raw_df.to_string(index=False, float_format="%.5f"))
print("\nL10 nbh-corrected:")
print(corrected_df.to_string(index=False, float_format="%.5f"))

L10 raw:
parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00839 0.00498 0.01170     0.00318  0.00000         True
  sigma_8 0.00265 0.00140 0.00531     0.00325  0.14540        False
    A_SN1 0.00337 0.00184 0.00616     0.00320  0.03380         True
   A_AGN1 0.00181 0.00094 0.00477     0.00325  0.41540        False
    A_SN2 0.00235 0.00129 0.00504     0.00321  0.21540        False
   A_AGN2 0.00122 0.00078 0.00427     0.00318  0.72520        False

L10 nbh-corrected:
parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00783 0.00624 0.01021     0.00322  0.00000         True
  sigma_8 0.00394 0.00187 0.00650     0.00321  0.01000         True
    A_SN1 0.00212 0.00105 0.00481     0.00322  0.29920        False
   A_AGN1 0.00174 0.00094 0.00478     0.00323  0.45360        False
    A_SN2 0.00230 0.00123 0.00502     0.00324  0.22820        False
   A_AGN2 0.00158 0.00082 0.00448     0.00321  0.53540        False


## 2. Multi-selection comparison (paper figure)

In [6]:
if len(all_tables) >= 2:

    fig, ax = plot_multi_selection(
        tables=list(all_tables.values()),
        labels=list(all_tables.keys()),
        focus_params=PARAMS,
        figsize=(9, 5),
    )

    fig.savefig(f"{FIG_DIR}/sensitivity_comparison.pdf", dpi=300)
    fig.savefig(f"{FIG_DIR}/sensitivity_comparison.png", dpi=150)
    plt.close(fig)
    print(f"\nSaved: {FIG_DIR}/sensitivity_comparison.pdf")
else:
    print(f"\n[skip] only {len(all_tables)} selection(s) available; need >= 2")


Saved: ../plots/sensitivity_comparison.pdf


## 3. LaTeX-ready table for the paper

In [7]:
if all_tables:

    # Pick the luminosity selection (main result)
    key = "L10" if "L10" in all_tables else list(all_tables.keys())[0]
    df = all_tables[key]

    def fmt_ci(row):
        return (
            f"${row['R_obs']:.4f}"
            f"_{{-{row['R_obs']-row['ci_lo']:.4f}}}"
            f"^{{+{row['ci_hi']-row['R_obs']:.4f}}}$"
        )

    def fmt_p(p):
        if p < 0.001:
            return "$< 0.001$"
        elif p < 0.01:
            return f"${p:.3f}$"
        else:
            return f"${p:.2f}$"

    print(f"\n\n% LaTeX table for {key} selection")
    print(r"\begin{table}")
    print(r"  \centering")
    print(r"  \caption{RMS sensitivity with bootstrap CI and")
    print(r"           permutation $p$-values for the")
    print(f"           {key} selection at $z \\approx 1$.}}")
    print(r"  \begin{tabular}{lccc}")
    print(r"    \hline\hline")
    print(r"    Parameter & $R_p$ (95\% CI) & $p$-value & Significant \\")
    print(r"    \hline")

    for _, row in df.sort_values("R_obs", ascending=False).iterrows():
        sig = r"\checkmark" if row["significant"] else "---"
        print(
            f"    {row['parameter']:10s} & "
            f"{fmt_ci(row):30s} & "
            f"{fmt_p(row['p_value']):12s} & "
            f"{sig} \\\\"
        )

    print(r"    \hline")
    print(r"  \end{tabular}")
    print(r"\end{table}")
else:
    print("[skip] no tables available")



% LaTeX table for L10 selection
\begin{table}
  \centering
  \caption{RMS sensitivity with bootstrap CI and
           permutation $p$-values for the
           L10 selection at $z \approx 1$.}
  \begin{tabular}{lccc}
    \hline\hline
    Parameter & $R_p$ (95\% CI) & $p$-value & Significant \\
    \hline
    Omega_m    & $0.0084_{-0.0034}^{+0.0033}$   & $< 0.001$    & \checkmark \\
    A_SN1      & $0.0034_{-0.0015}^{+0.0028}$   & $0.03$       & \checkmark \\
    sigma_8    & $0.0027_{-0.0013}^{+0.0027}$   & $0.15$       & --- \\
    A_SN2      & $0.0024_{-0.0011}^{+0.0027}$   & $0.22$       & --- \\
    A_AGN1     & $0.0018_{-0.0009}^{+0.0030}$   & $0.42$       & --- \\
    A_AGN2     & $0.0012_{-0.0004}^{+0.0031}$   & $0.73$       & --- \\
    \hline
  \end{tabular}
\end{table}


In [8]:
raw_df_fedd, corrected_df_fedd, theta_resid_fedd = nbh_corrected_sensitivity(
    residuals=result_fedd["residuals"],
    theta=result_fedd["theta"],
    nbh=result_fedd["nbh"],
    params=PARAMS,
    n_boot=N_BOOT, n_null=N_NULL,
)

print("fEdd10 raw:")
print(raw_df_fedd.to_string(index=False, float_format="%.5f"))
print("\nfEdd10 nbh-corrected:")
print(corrected_df_fedd.to_string(index=False, float_format="%.5f"))

fEdd10 raw:
parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00899 0.00539 0.01242     0.00318  0.00000         True
  sigma_8 0.00191 0.00131 0.00489     0.00323  0.36720        False
    A_SN1 0.00493 0.00334 0.00750     0.00323  0.00060         True
   A_AGN1 0.00210 0.00095 0.00501     0.00321  0.28620        False
    A_SN2 0.00284 0.00151 0.00556     0.00322  0.09840        False
   A_AGN2 0.00132 0.00095 0.00436     0.00316  0.67500        False

fEdd10 nbh-corrected:
parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00825 0.00674 0.01055     0.00319  0.00000         True
  sigma_8 0.00394 0.00184 0.00654     0.00321  0.01080         True
    A_SN1 0.00403 0.00241 0.00660     0.00325  0.00740         True
   A_AGN1 0.00194 0.00094 0.00498     0.00321  0.35800        False
    A_SN2 0.00288 0.00153 0.00561     0.00320  0.08840        False
   A_AGN2 0.00174 0.00095 0.00463     0.00319  0.44400        False


In [9]:
# Negative-control check. M1e6 keeps every BH above the mass threshold
# (no activity-ranked top-10% cut), so there's no a priori reason for
# its nbh to be entangled with theta the way L10/fEdd10's is. Diagnose
# first: if M1e6 *does* show a strong nbh-theta correlation, this isn't
# a clean control -- it just means M1e6 needs correcting too, which is
# useful to know but doesn't test for a residualization artifact. If it
# comes back clean but the corrected table still swings a lot, that
# points to a quartile-boundary artifact in the residualization itself.

result_m1e6 = load_and_analyze(
    SELECTIONS["M1e6"], params=PARAMS, params_file=PARAM_FILE,
)

m1e6_diag = diagnose_activity_cut_bias(
    sim_ids=result_m1e6["sim_ids"], nbh=result_m1e6["nbh"],
    theta_all=theta_all, params=PARAMS,
)
print("M1e6 nbh-vs-theta correlation:")
print(m1e6_diag.to_string(index=False))

raw_df_m1e6, corrected_df_m1e6, theta_resid_m1e6 = nbh_corrected_sensitivity(
    residuals=result_m1e6["residuals"],
    theta=result_m1e6["theta"],
    nbh=result_m1e6["nbh"],
    params=PARAMS,
    n_boot=N_BOOT, n_null=N_NULL,
)

print("\nM1e6 raw:")
print(raw_df_m1e6.to_string(index=False, float_format="%.5f"))
print("\nM1e6 nbh-corrected:")
print(corrected_df_m1e6.to_string(index=False, float_format="%.5f"))

M1e6 nbh-vs-theta correlation:
parameter  spearman_rho  spearman_p  bias_flag
  Omega_m      0.980290    0.000000       True
  sigma_8      0.143823    0.000005       True
    A_SN1     -0.078359    0.013189      False
   A_AGN1      0.003304    0.916883      False
    A_SN2      0.018613    0.556597      False
   A_AGN2     -0.007192    0.820314      False


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(



M1e6 raw:
parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00725 0.00420 0.01008     0.00261  0.00000         True
  sigma_8 0.00492 0.00321 0.00686     0.00262  0.00000         True
    A_SN1 0.00435 0.00267 0.00659     0.00258  0.00020         True
   A_AGN1 0.00221 0.00089 0.00439     0.00263  0.11400        False
    A_SN2 0.00191 0.00087 0.00414     0.00263  0.21680        False
   A_AGN2 0.00057 0.00049 0.00297     0.00257  0.91780        False

M1e6 nbh-corrected:
parameter   R_obs   ci_lo   ci_hi  null_floor  p_value  significant
  Omega_m 0.00628 0.00449 0.00828     0.00260  0.00000         True
  sigma_8 0.00564 0.00371 0.00745     0.00260  0.00000         True
    A_SN1 0.00469 0.00293 0.00649     0.00260  0.00000         True
   A_AGN1 0.00211 0.00087 0.00439     0.00261  0.13960        False
    A_SN2 0.00186 0.00084 0.00409     0.00261  0.22740        False
   A_AGN2 0.00084 0.00050 0.00311     0.00259  0.77060        False


In [10]:
# True negative control: permute nbh across sims so it has zero real
# relationship to theta while keeping its marginal distribution intact,
# then rerun the correction on the shuffled nbh. If a *fake* nbh-theta
# relationship still moves R_obs/significance around as much as the
# real nbh did for L10/fEdd10, that points to a quartile-boundary
# artifact in the correction itself rather than a genuine confound.
# Scoped to the two activity selections only.

N_PERM = 10
PERM_N_BOOT, PERM_N_NULL = 1000, 1000  # reduced for this diagnostic only -- not for the paper table

rng = np.random.default_rng(7)

def permuted_corrected_runs(residuals, theta, nbh, params, n_perm, n_boot, n_null, rng):
    records = []
    nbh = np.asarray(nbh)
    for i in range(n_perm):
        nbh_shuffled = rng.permutation(nbh)
        _, fake_corrected_df, _ = nbh_corrected_sensitivity(
            residuals=residuals, theta=theta, nbh=nbh_shuffled,
            params=params, n_boot=n_boot, n_null=n_null,
        )
        fake_corrected_df["perm"] = i
        records.append(fake_corrected_df)
    return pd.concat(records, ignore_index=True)

print("L10 permutation check (10 fake nbh draws):")
perm_df_l10 = permuted_corrected_runs(
    result["residuals"], result["theta"], result["nbh"],
    PARAMS, N_PERM, PERM_N_BOOT, PERM_N_NULL, rng,
)
print(perm_df_l10.groupby("parameter").agg(
    R_obs_mean=("R_obs", "mean"),
    R_obs_std=("R_obs", "std"),
    frac_significant=("significant", "mean"),
))

print("\nfEdd10 permutation check (10 fake nbh draws):")
perm_df_fedd = permuted_corrected_runs(
    result_fedd["residuals"], result_fedd["theta"], result_fedd["nbh"],
    PARAMS, N_PERM, PERM_N_BOOT, PERM_N_NULL, rng,
)
print(perm_df_fedd.groupby("parameter").agg(
    R_obs_mean=("R_obs", "mean"),
    R_obs_std=("R_obs", "std"),
    frac_significant=("significant", "mean"),
))

L10 permutation check (10 fake nbh draws):
           R_obs_mean  R_obs_std  frac_significant
parameter                                         
A_AGN1       0.001979   0.000408               0.0
A_AGN2       0.001388   0.000150               0.0
A_SN1        0.003580   0.000183               1.0
A_SN2        0.002291   0.000193               0.0
Omega_m      0.008359   0.000129               1.0
sigma_8      0.002568   0.000138               0.0

fEdd10 permutation check (10 fake nbh draws):
           R_obs_mean  R_obs_std  frac_significant
parameter                                         
A_AGN1       0.002120   0.000298               0.0
A_AGN2       0.001432   0.000137               0.0
A_SN1        0.005165   0.000238               1.0
A_SN2        0.002889   0.000136               0.0
Omega_m      0.008956   0.000065               1.0
sigma_8      0.001941   0.000145               0.0


## 4. Redshift evolution with error bars

Labels are derived from `SNAP_TO_Z` keyed by snapshot number, so they
stay correct regardless of which snapshots are present/missing
(the old version assumed `SNAPS` always had exactly snaps
32/50/90 present, in that order, which silently mislabels the
figure if any snapshot is skipped).

In [11]:
SNAPS = {snap: f"{OUTPUT_DIR}/knn_snap{snap}_M1e+06.npz" for snap in SNAP_TO_Z}

snap_tables = {}
for snap, path in SNAPS.items():
    try:
        result = load_and_analyze(path, params=PARAMS, params_file=PARAM_FILE)
        snap_tables[snap] = sensitivity_table(
            result["residuals"], result["theta"],
            params=PARAMS,
            n_boot=N_BOOT,
            n_null=N_NULL,
        )
        print(f"\nSnap {snap} (z~{SNAP_TO_Z[snap]}): done")
    except FileNotFoundError:
        print(f"\nSnap {snap}: [skip] not found")

if len(snap_tables) >= 2:
    snaps_present = list(snap_tables.keys())
    labels = [f"z≈{SNAP_TO_Z[snap]}" for snap in snaps_present]

    fig, ax = plot_multi_selection(
        tables=list(snap_tables.values()),
        labels=labels,
        focus_params=["Omega_m", "sigma_8", "A_SN1"],
        figsize=(7, 4.5),
    )
    ax.set_title("Redshift evolution of parameter sensitivity")
    fig.savefig(f"{FIG_DIR}/sensitivity_vs_redshift.pdf", dpi=300)
    plt.close(fig)
    print(f"Saved: {FIG_DIR}/sensitivity_vs_redshift.pdf")
else:
    print(f"\n[skip] only {len(snap_tables)} snapshot(s) available; need >= 2")

print("\n\nDone.")

/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(



Snap 32 (z~3): done


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(



Snap 50 (z~1): done


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(



Snap 90 (z~0): done
Saved: ../plots/sensitivity_vs_redshift.pdf


Done.
